# Comparisons figures

Use the bao kernel in VS Code in your allocated compute session. Choose saved inputs below, then edit and run the plot cells you need. Leave an input as `None` to skip that section.


In [ ]:
from pathlib import Path
import csv
import json
import os
import sys
import itertools

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pzbao/run.py").is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from pzbao.figures import data as readers, style

BINS, TIERS, STATES, PARAMETERS = readers.BINS, readers.TIERS, readers.STATES, readers.PARAMETERS
LABELS, ROBUSTNESS = readers.LABELS, readers.ROBUSTNESS
figures = {}


def show(fig, name, caption=""):
    figures[name] = fig
    display(fig)
    if caption:
        display(Markdown(caption))
    plt.close(fig)


In [ ]:
TIER_COLORS = ("#4d4d4d", "#1f77b4", "#2ca02c", "#ff7f00", "#e41a1c")
# Colors read from the SVG of Paillas et al. 2404.03005v3, Figure 15.
BIN_COLORS = dict(zip(BINS, ("#9acd32", "#9acd32", "#ffa500", "#ff4500", "#b22222")))
ZLABELS = (
    r"$0.1\leq z<0.2$",
    r"$0.2\leq z<0.4$",
    r"$0.4\leq z<0.6$",
    r"$0.6\leq z<0.8$",
    r"$0.8\leq z<1.0$",
)
VARIANTS = ("baseline", "pre", "pk", "ngc", "flat", "pcs2", "reciso", "r25")
VARIANT_LABELS = (
    "Baseline: power3, RecSym, R15",
    "Pre-reconstruction",
    "Power spectrum only",
    "NGC only",
    "Flat damping priors",
    "PCS2 spline broadband",
    "RecIso",
    r"$R=25\,h^{-1}\mathrm{Mpc}$",
)
LABELS = {"qiso": r"$\alpha_{\rm iso}$", "qap": r"$\alpha_{\rm AP}$"}


def presentation_style():
    import matplotlib as mpl

    mpl.rcParams.update(
        {
            "font.family": "serif",
            "mathtext.fontset": "cm",
            "font.size": 11,
            "axes.labelsize": 13,
            "axes.titlesize": 12,
            "axes.linewidth": 1.2,
            "xtick.direction": "in",
            "ytick.direction": "in",
            "xtick.top": True,
            "ytick.right": True,
            "savefig.dpi": 200,
        }
    )


def panel_grid():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 3, figsize=(13.5, 8), layout="constrained")
    axes = list(axes.flat)
    axes[-1].axis("off")
    return fig, axes


presentation_style()

from pzbao.figures.style import sigma_color, two_col


## Saved comparison tables

Choose a saved report for each section. Comparison reports contain fit and robustness tables; LRG reports also contain the saved fit curves. Missing or flagged results remain labeled in the plots.


In [ ]:
COMPARISON_REPORT = None
DENSITY_REPORT = None
JOINT_FOM_EXPORT = None  # identified figure-data.json from the native posterior diagnostics
LRG_REPORTS = []  # Native saved LRG ensemble bundles

comparison = {}
if COMPARISON_REPORT is not None:
    comparison = readers.read_tables(COMPARISON_REPORT, ["robustness.json"])


In [ ]:
def r25_plot_design(rows):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FixedLocator

    result = []
    for parameter in readers.PARAMETERS:
        limits = readers.COMPARISON_XLIM[parameter]
        fig, axes = plt.subplots(1, 5, figsize=(16, 5.4), sharex=True, layout="constrained")
        for ax, name in zip(axes, readers.BINS):
            group = readers.available_rows(rows, name, parameter)
            color = BIN_COLORS[name]
            baseline = next(r for r in group if r["variant"] == "baseline")
            if baseline["usable"]:
                ax.axvspan(*baseline["minos_bounds"], color=color, alpha=0.12)
            for y, row in enumerate(group):
                value = row["value"]
                lo = hi = value
                if row["ordered_bounds"]:
                    lo, hi = row["minos_bounds"]
                    ax.errorbar(
                        value,
                        y,
                        xerr=[[value - lo], [hi - value]],
                        fmt="o" if row["usable"] else "x",
                        color=color,
                        capsize=3,
                        lw=1.4,
                        ms=5,
                    )
                else:
                    ax.plot(value, y, "x", color=color, ms=6)
                if min(lo, value) < limits[0]:
                    ax.plot(limits[0], y, "<", color=color, ms=6, clip_on=False)
                if max(hi, value) > limits[1]:
                    ax.plot(limits[1], y, ">", color=color, ms=6, clip_on=False)
            ax.set(
                xlim=limits,
                ylim=(len(group) - 0.4, -0.6),
                xlabel=readers.LABELS[parameter],
                title=name,
                yticks=range(len(group)),
                yticklabels=[readers.R25_VARIANTS[r["variant"]] for r in group],
            )
            ax.axvline(1, color=".6", ls=":", lw=0.9)
            ax.xaxis.set_major_locator(
                FixedLocator(np.linspace(*limits, 5 if parameter == "qiso" else 4))
            )
            ax.tick_params(labelsize=9)
            ax.minorticks_on()
        fig.suptitle(r"$\sigma_z/(1+z)=0.003$: mean of 25 mocks, $C/25$; 68% profile intervals")
        r25_bins = [
            name
            for name in readers.BINS
            if any(r["variant"] == "r25" for r in readers.available_rows(rows, name, parameter))
        ]
        show(
            fig,
            "design-" + parameter,
            "Saved native mean fits, with the existing tracer palette and baseline band. "
            "Only available rows are shown in each panel; crosses retain flagged fits. "
            f"R25 is available for {', '.join(r25_bins) or 'no bins'} and holds the R15 damping priors fixed. "
            f"All panels span {limits[0]:g} to {limits[1]:g}; edge triangles mark intervals or "
            "centres outside this range, with full values retained in the tables. "
            "P(k), NGC-only and RecIso are set aside. Cap convergence warnings remain "
            "in r25-diagnostics.json; numerical success is not a validation verdict.",
        )
    return result


if comparison:
    r25_plot_design(comparison["robustness.json"])


## Full comparison inventory

The full panel retains unavailable slots, including P(k), NGC and RecIso. The focused panel above shows available baseline/pre/flat/PCS2/R25 rows.


In [ ]:
def claim_figures_plot_robustness(rows):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator

    for parameter in PARAMETERS:
        extent = [1.0]
        for row in rows:
            if row["parameter"] == parameter and row["status"] == "available":
                extent.extend(row["minos_bounds"] if row["ordered_bounds"] else [row["value"]])
        lower, upper = min(extent), max(extent)
        padding = 0.08 * (upper - lower) if upper > lower else 0.05
        fig, axes = plt.subplots(1, 5, figsize=(13, 4.5), sharey=True, layout="constrained")
        for ax, bin_name in zip(axes, BINS):
            group = {
                r["variant"]: r
                for r in rows
                if r["measurement_bin"] == bin_name and r["parameter"] == parameter
            }
            baseline = group["baseline"]
            if baseline["status"] == "available" and baseline["usable"]:
                ax.axvspan(*baseline["minos_bounds"], color="C0", alpha=0.12)
            ax.axvline(1, color="0.5", lw=0.8, ls="--")
            for y, (variant, _) in enumerate(ROBUSTNESS):
                r = group[variant]
                if r["status"] == "missing":
                    ax.text(
                        0.97,
                        y,
                        "missing",
                        ha="right",
                        va="center",
                        transform=ax.get_yaxis_transform(),
                        color="0.6",
                        fontsize=8,
                    )
                    continue
                color = "C0" if r["usable"] else "#b34b20"
                if r["ordered_bounds"]:
                    lo, hi = r["minos_bounds"]
                    ax.hlines(y, lo, hi, color=color, ls="-" if r["usable"] else ":", lw=1.4)
                ax.plot(r["value"], y, "o" if r["usable"] else "x", color=color, ms=4)
            ax.set(title=bin_name, xlabel=LABELS[parameter], ylim=(7.6, -0.6))
            ax.set_xlim(lower - padding, upper + padding)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.tick_params(axis="x", labelsize=8)
        axes[0].set_yticks(range(len(ROBUSTNESS)), [label for _, label in ROBUSTNESS], fontsize=9)
        fig.suptitle(
            r"$\sigma_z/(1+z)=0.003$: mean of 25 mocks, $C/25$; 68% profile intervals",
            fontsize=11,
        )
        show(
            fig,
            f"robustness-{parameter}",
            "Blue band: usable baseline interval. Orange crosses/dotted bounds: "
            "flagged fits. Missing is not zero. No posterior ESS is implied.",
        )


if comparison:
    claim_figures_plot_robustness(comparison["robustness.json"])


In [ ]:
def claim_figures_plot_recovery(summary, pairs, tiers):
    import matplotlib.pyplot as plt

    ticks = ["native" if t == "native" else t.removeprefix("sigma_") for t in tiers]
    for parameter in PARAMETERS:
        fig, axes = plt.subplots(
            2, 5, figsize=(13, 5.8), sharex=True, sharey="row", layout="constrained"
        )
        for col, bin_name in enumerate(BINS):
            ax = axes[0, col]
            for state, shift, color in zip(STATES, (-0.12, 0.12), ("0.45", "C0")):
                for x, tier in enumerate(tiers):
                    row = next(
                        r
                        for r in summary
                        if (r["measurement_bin"], r["parameter"], r["condition"], r["state"])
                        == (bin_name, parameter, tier, state)
                    )
                    if row["bias"] is not None:
                        ax.errorbar(
                            x + shift,
                            100 * row["bias"],
                            yerr=100 * row["bias_sem"] if row["bias_sem"] is not None else None,
                            fmt="o",
                            ms=3,
                            color=color,
                        )
            ax.axhline(0, lw=0.8, color="0.5")
            ax.set_title(bin_name)
            if not any(r["n_saved"] for r in summary if r["measurement_bin"] == bin_name):
                ax.text(
                    0.5,
                    0.5,
                    "Individual fits missing",
                    transform=ax.transAxes,
                    ha="center",
                    fontsize=8,
                )
            lower = axes[1, col]
            lower.axhline(0, lw=0.8, color="0.5")
            for x, tier in enumerate(tiers):
                group = [
                    r
                    for r in pairs
                    if (r["measurement_bin"], r["condition"], r["parameter"])
                    == (bin_name, tier, parameter)
                ]
                gains = [r["profile_width_gain_percent"] for r in group if r["usable_intervals"]]
                if gains:
                    lower.scatter(np.full(len(gains), x), gains, s=7, alpha=0.25, color="C0")
                    lower.plot(x, np.median(gains), "_", color="k", ms=12)
                lower.text(
                    x,
                    0.98,
                    f"{len(gains)}/25",
                    transform=lower.get_xaxis_transform(),
                    ha="center",
                    va="top",
                    fontsize=7,
                )
            lower.set_xticks(range(len(tiers)), ticks, rotation=45, fontsize=8)
            lower.set_xlabel(r"$\sigma_z/(1+z)$")
        axes[0, 0].set_ylabel(r"$100(\langle\alpha\rangle-1)$")
        axes[1, 0].set_ylabel("Profile-width gain [%]")
        fig.suptitle(
            LABELS[parameter] + ": grey pre, blue post; bias bars are mock SEM", fontsize=11
        )
        show(
            fig,
            f"recovery-{parameter}",
            "All saved converged best fits enter the bias, including flagged intervals. "
            "Gain uses paired usable intervals only; counts show inclusion. Dots are "
            "mocks and black ticks medians. Profile-width gain is not the joint FoM.",
        )


RECOVERY_REPORT = None
if RECOVERY_REPORT is not None:
    saved = readers.read_tables(RECOVERY_REPORT, ["fit-diagnostics.json"])
    recovery = saved["fit-diagnostics.json"]
    summary, pairs = readers.summarize_recovery(recovery, TIERS)
    claim_figures_plot_recovery(summary, pairs, TIERS)


In [ ]:
def claim_figures_plot_components(pairs):
    import matplotlib.pyplot as plt

    for pair in pairs:
        ref = pair["power3"]
        s = np.asarray(ref["separation"])
        fig, axes = plt.subplots(
            4,
            len(ref["multipoles"]),
            figsize=(8, 8),
            squeeze=False,
            sharex=True,
            layout="constrained",
        )
        for col, ell in enumerate(ref["multipoles"]):
            sl = slice(col * len(s), (col + 1) * len(s))
            data = np.asarray(ref["residuals"]["data"])[sl]
            sd = np.asarray(ref["standard_deviation"])[sl]
            axes[0, col].errorbar(s, s**2 * data, yerr=s**2 * sd, fmt="k.", ms=3)
            for model, color in (("power3", "C0"), ("pcs2", "C1")):
                c = pair[model]["components"]
                for row, field in ((0, "model"), (1, "broadband"), (2, "bao_template")):
                    axes[row, col].plot(
                        s, s**2 * np.asarray(c[field])[sl], color=color, label=model
                    )
                axes[2, col].plot(s, s**2 * np.asarray(c["bao_data"])[sl], ".", color=color, ms=3)
                axes[3, col].plot(s, (data - np.asarray(c["model"])[sl]) / sd, color=color)
            axes[0, col].set_title(rf"$\ell={ell}$")
            axes[3, col].axhline(0, color="0.5", lw=0.7)
            axes[3, col].set_xlabel(r"$s\ [h^{-1}\mathrm{Mpc}]$")
        labels = (
            r"$s^2\xi$: total",
            r"$s^2 B$: broadband",
            r"$s^2\xi$: acoustic",
            r"Residual / $\sigma$",
        )
        for ax, label in zip(axes[:, 0], labels):
            ax.set_ylabel(label)
        axes[0, -1].legend(fontsize=8)
        name = f"broadband-{pair['measurement_bin']}-{pair['state']}"
        fig.suptitle(f"{pair['measurement_bin']}, sigma_0.003, {pair['state']}", fontsize=10)
        show(
            fig,
            name,
            "Matched saved data, covariance and physical priors. Acoustic points "
            "subtract each model's smooth template and broadband. Diagonal-normalized "
            "residuals remain correlated. Mean-fit shifts alone do not establish an "
            "ensemble bias reduction.",
        )


COMPONENT_REPORT = None
if COMPONENT_REPORT is not None:
    saved = readers.read_tables(COMPONENT_REPORT, ["broadband-components.json"])
    components = saved["broadband-components.json"]
    claim_figures_plot_components(components)


In [ ]:
def saved_comparisons_plot_density(rows):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator

    for parameter in readers.PARAMETERS:
        fig, axes = plt.subplots(
            1, 2, figsize=(8, 3.4), sharey=True, sharex=True, layout="constrained"
        )
        for ax, name in zip(axes, ("LRG2", "LRG3")):
            for y, (state, variant) in enumerate(
                itertools.product(readers.STATES, ("baseline", "doubled"))
            ):
                row = next(
                    r
                    for r in rows
                    if r["measurement_bin"] == name
                    and r["parameter"] == parameter
                    and r["reconstruction_state"] == state
                    and r["variant"] == variant
                )
                color = "C0" if variant == "doubled" else "0.45"
                if row["ordered_bounds"]:
                    ax.hlines(
                        y, *row["minos_bounds"], color=color, ls="-" if row["usable"] else ":"
                    )
                ax.plot(row["value"], y, "o" if row["usable"] else "x", color=color)
            ax.axvline(1, color="0.6", ls="--", lw=0.8)
            ax.set(title=name, xlabel=readers.LABELS[parameter], ylim=(3.6, -0.6))
            ax.xaxis.set_major_locator(MaxNLocator(4))
        axes[0].set_yticks(
            range(4),
            ["Pre: baseline", "Pre: doubled density", "Post: baseline", "Post: doubled density"],
        )
        fig.suptitle(r"Density comparison, $\sigma_z/(1+z)=0.001$: mean of 25, $C/25$", fontsize=11)
        show(
            fig,
            f"density-control-{parameter}",
            "Matched power3 mean fits with each sample's covariance and fixed reference damping priors. "
            "Crosses/dotted bounds retain unusable intervals. Mean-fit shifts do not measure ensemble bias. "
            "Individual density fits remain incomplete; no joint posterior FoM is inferred here.",
        )


if DENSITY_REPORT is not None:
    saved = readers.read_tables(DENSITY_REPORT, ["density-control.json"])
    saved_comparisons_plot_density(saved["density-control.json"])


In [ ]:
def joint_fom_figure(records):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter

    pairs = readers.paired_values(records)
    lookup = {readers.route_key(r["route"]): r for r in records}
    fig, axes = plt.subplots(
        2, 5, figsize=(13, 6), constrained_layout=True, gridspec_kw={"height_ratios": [2, 1]}
    )
    x = np.arange(len(TIERS))
    for column, name in enumerate(BINS):
        top, bottom = axes[:, column]
        group = [p for p in pairs if p["measurement_bin"] == name]
        for state, field, offset, colour in (
            (STATES[0], "pre", -0.18, "#8a8a8a"),
            (STATES[1], "post", 0.18, "#0072B2"),
        ):
            for position, row in enumerate(group):
                value = row[field]
                coordinate = position + offset
                if value is None:
                    top.text(
                        coordinate,
                        0.02,
                        "NA",
                        rotation=90,
                        ha="center",
                        va="bottom",
                        fontsize=6,
                        transform=top.get_xaxis_transform(),
                    )
                    continue
                top.bar(
                    coordinate,
                    value,
                    width=0.34,
                    color=colour,
                    label=field if position == 0 else None,
                )
                source = lookup["--".join((name, row["condition"], state))]
                if source["profile_flagged"]:
                    top.plot(coordinate, value, "kx", ms=5)
                if source["sampling_limits"]:
                    top.annotate(
                        "†",
                        (coordinate, value),
                        xytext=(0, 5),
                        textcoords="offset points",
                        ha="center",
                        fontsize=8,
                    )
                if source.get("covariance_validity", {}).get("structural_test_warnings"):
                    top.annotate(
                        "‡",
                        (coordinate, value),
                        xytext=(5, 5),
                        textcoords="offset points",
                        ha="center",
                        fontsize=8,
                    )
        top.set_yscale("log")
        finite = [row[k] for row in group for k in ("pre", "post") if row[k] is not None]
        top.set_ylim((min(finite) / 1.8, max(finite) * 2) if finite else (1, 10))
        top.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
        top.set_title(name)
        gains = [row["gain"] if row["gain"] is not None else np.nan for row in group]
        bottom.plot(x, gains, "o-", color="#0072B2", ms=4)
        bottom.axhline(1, color=".5", lw=0.8)
        for position, row in enumerate(group):
            if row["gain"] is None:
                bottom.text(
                    position,
                    0.04,
                    "NA",
                    transform=bottom.get_xaxis_transform(),
                    ha="center",
                    fontsize=7,
                )
            else:
                members = [lookup["--".join((name, row["condition"], state))] for state in STATES]
                if any(r["profile_flagged"] for r in members):
                    bottom.plot(position, row["gain"], "kx", ms=6)
                if any(r["sampling_limits"] for r in members):
                    bottom.annotate(
                        "†",
                        (position, row["gain"]),
                        xytext=(3, 4),
                        textcoords="offset points",
                        fontsize=8,
                    )
        for axis in (top, bottom):
            axis.set_xticks(x, ["native", ".001", ".003", ".01", ".03"], rotation=50)
            axis.tick_params(labelsize=8)
            axis.set_xlim(-0.6, len(TIERS) - 0.4)
        bottom.set_xlabel(r"$\sigma_z/(1+z)$")
    axes[0, 0].set_ylabel("Absolute joint FoM (log scale)")
    axes[1, 0].set_ylabel("FoM post / pre")
    from matplotlib.patches import Patch

    fig.legend(
        handles=[
            Patch(color="#8a8a8a", label="Pre-reconstruction"),
            Patch(color="#0072B2", label="Post-reconstruction"),
        ],
        loc="upper center",
        bbox_to_anchor=(0.5, 1.025),
        ncol=2,
    )
    count = sum(r["status"] == "available" for r in records)
    fig.suptitle(
        f"Power3 mean-data posterior: {count}/50 routes"
        + (" — partial grid" if count < 50 else ""),
        fontsize=12,
        y=1.08,
    )
    fig.text(
        0.5,
        -0.04,
        "× invalid/boundary-limited or unavailable profile interval; "
        "† sampling limitation (including unavailable ESS); ‡ covariance warning; NA unavailable.\n"
        "Joint FoM = 1/√det Cov(αiso, αAP), ddof=1. Conditional mean covariance; "
        "numerical stopping is not a validation verdict.",
        ha="center",
        fontsize=8,
    )
    return fig, pairs


if JOINT_FOM_EXPORT is not None:
    records = readers.read_json(JOINT_FOM_EXPORT)["records"]
    fig, pairs = joint_fom_figure(records)
    show(fig, "power3-joint-fom")


In [ ]:
if LRG_REPORTS:
    lrg = readers.load_lrg_reports(LRG_REPORTS)
    lrg_rows, lrg_pairs, lrg_stats = lrg["rows"], lrg["pairs"], lrg["summary"]
    curves, lrg_grid = lrg["curves"], lrg["grid"]
    plt.rcParams.update(
        {
            "font.size": 8,
            "axes.titlesize": 8,
            "legend.fontsize": 7,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )


In [ ]:
def lrg_figures_primary_figures(rows, pairs, stats, grid):
    conditions = grid["sigma_z_tiers"]
    labels = [c.replace("sigma_", "") if c != "native" else "0" for c in conditions]
    colours = ("#737e90", "#087e8b")
    for quantity, ylabel, caption in (
        (
            "bias",
            r"$\langle\hat\alpha\rangle-1$",
            "Mean of all fitted centres; bars are ensemble SEM.",
        ),
        (
            "scatter",
            r"$s(\hat\alpha)$",
            "Sample scatter of all centres (ddof=1), not posterior width.",
        ),
        (
            "median_width",
            "MINOS half-width",
            "Median valid, unconfined profile width; crosses below retain flagged widths.",
        ),
        (
            "median_precision",
            "Inverse MINOS half-width",
            "Absolute one-dimensional precision, median of valid unconfined inverse widths.",
        ),
    ):
        fig, axes = plt.subplots(2, 3, figsize=two_col(4.5), constrained_layout=True)
        for p, parameter in enumerate(PARAMETERS):
            for b, name in enumerate(readers.LRG_BINS):
                ax = axes[p, b]
                for state, colour, offset in zip(STATES, colours, (-0.08, 0.08)):
                    selected = [
                        next(
                            s
                            for s in stats
                            if (s["measurement_bin"], s["condition"], s["parameter"], s["state"])
                            == (name, c, parameter, state)
                        )
                        for c in conditions
                    ]
                    y = [s[quantity] if s[quantity] is not None else np.nan for s in selected]
                    ax.errorbar(
                        np.arange(len(conditions)) + offset,
                        y,
                        yerr=[s["sem"] for s in selected] if quantity == "bias" else None,
                        fmt="o-",
                        ms=3,
                        lw=0.8,
                        color=colour,
                        label=state[:4],
                    )
                    if quantity == "median_width":
                        for x, c in enumerate(conditions):
                            bad = [
                                lrg_interval(r)["width"]
                                for r in rows
                                if (
                                    r["measurement_bin"],
                                    r["condition"],
                                    r["parameter"],
                                    r["state"],
                                )
                                == (name, c, parameter, state)
                                and not lrg_interval(r)["usable"]
                                and lrg_interval(r)["width"] is not None
                            ]
                            ax.scatter(
                                [x + offset] * len(bad),
                                bad,
                                marker="x",
                                s=10,
                                color=colour,
                                alpha=0.4,
                            )
                ax.set(
                    title=rf"{name} / {parameter}", xticks=range(len(labels)), xticklabels=labels
                )
                ax.set_ylabel(ylabel)
                if quantity == "bias":
                    ax.axhline(0, color="0.7", lw=0.7)
                if p == 1:
                    ax.set_xlabel(r"$\sigma_z/(1+z)$")
        axes[0, 0].legend(fontsize=7)
        show(
            fig,
            "primary-" + quantity,
            caption + " All failures and prior contact remain in the source table.",
        )

    fig, axes = plt.subplots(2, 3, figsize=two_col(4.8), constrained_layout=True)
    for p, parameter in enumerate(PARAMETERS):
        for b, name in enumerate(readers.LRG_BINS):
            ax = axes[p, b]
            for x, c in enumerate(conditions):
                selected = [
                    r
                    for r in pairs
                    if (r["measurement_bin"], r["condition"], r["parameter"])
                    == (name, c, parameter)
                ]
                for usable, marker in ((True, "o"), (False, "x")):
                    values = [
                        r["raw_width_ratio"]
                        for r in selected
                        if r["usable"] == usable and r["raw_width_ratio"] is not None
                    ]
                    ax.scatter(
                        np.linspace(x - 0.12, x + 0.12, len(values)),
                        values,
                        marker=marker,
                        s=10,
                        alpha=0.55,
                        color="#087e8b" if usable else "#a23c3c",
                    )
                good = [r["uncertainty_ratio"] for r in selected if r["usable"]]
                if good:
                    ax.plot([x - 0.2, x + 0.2], [np.median(good)] * 2, color="black", lw=1)
                ax.text(
                    x,
                    0.98,
                    f"{len(good)}/{len(selected)}",
                    transform=ax.get_xaxis_transform(),
                    ha="center",
                    va="top",
                    fontsize=6,
                )
            ax.axhline(1, color="0.6", ls=":")
            ax.set(
                title=rf"{name} / {parameter}",
                xticks=range(len(labels)),
                xticklabels=labels,
                ylabel=r"$\sigma_{\rm post}/\sigma_{\rm pre}$",
            )
            if p == 1:
                ax.set_xlabel(r"$\sigma_z/(1+z)$")
    show(
        fig,
        "primary-gain",
        "Paired individual MINOS half-width ratios; improvement below one. "
        "Dots and black medians use valid, unconfined pairs with valid optimizers. Red crosses "
        "are flagged numerical ratios, not reliable gains. Labels give usable/all pairs.",
    )

    fig, axes = plt.subplots(2, 3, figsize=two_col(4.5), constrained_layout=True)
    for p, parameter in enumerate(PARAMETERS):
        for b, name in enumerate(readers.LRG_BINS):
            selected = [
                next(
                    s
                    for s in stats
                    if (s["measurement_bin"], s["condition"], s["parameter"], s["state"])
                    == (name, c, parameter, STATES[1])
                )
                for c in conditions
            ]
            axes[p, b].errorbar(
                range(len(conditions)),
                [s["paired_shift"] for s in selected],
                yerr=[s["paired_shift_sem"] for s in selected],
                fmt="o",
                ms=3,
            )
            axes[p, b].axhline(0, color="0.7", lw=0.7)
            axes[p, b].set(
                title=rf"{name} / {parameter}",
                xticks=range(len(labels)),
                xticklabels=labels,
                ylabel=r"$\langle\hat\alpha_{\rm post}-\hat\alpha_{\rm pre}\rangle$",
            )
    show(
        fig,
        "primary-paired-shifts",
        "Same-mock post-minus-pre fitted-centre shifts for every "
        "realization, including flagged fits. Bars are the SEM of paired differences, not "
        "the quadrature sum of independent pre/post SEMs. Flags remain in paired.json.",
    )


if LRG_REPORTS:
    lrg_figures_primary_figures(lrg_rows, lrg_pairs, lrg_stats, lrg_grid)


In [ ]:
def lrg_figures_curve_figures(curves, grid):
    source = {}
    for condition in grid["sigma_z_tiers"]:
        fig, axes = plt.subplots(2, 3, figsize=two_col(4.6), constrained_layout=True)
        residual_fig, residual_axes = plt.subplots(
            2, 3, figsize=two_col(4.6), constrained_layout=True
        )
        for b, name in enumerate(readers.LRG_BINS):
            for state, colour, linestyle in zip(STATES, ("#737e90", "#087e8b"), ("--", "-")):
                group = readers.lrg_clustering(curves, name, condition, state, grid["mock_ids"])
                source[f"{name}--{condition}--{state}"] = {
                    k: v.tolist() for k, v in group.items() if k != "example"
                }
                s = group["separation"]
                for p, ell in enumerate(group["multipoles"]):
                    block = slice(p * len(s), (p + 1) * len(s))
                    mean, scatter = s**2 * group["mean"][block], s**2 * group["scatter"][block]
                    ax = axes[p, b]
                    ax.plot(s, mean, color=colour, ls=linestyle, label=state[:4])
                    ax.fill_between(s, mean - scatter, mean + scatter, color=colour, alpha=0.1)
                    ax.set(
                        title=rf"{name} / $\ell={ell}$",
                        ylabel=r"$s^2\xi_\ell$ [$(h^{-1}{\rm Mpc})^2$]",
                    )
                    curve = group["example"]
                    residual_axes[p, b].plot(
                        curve["separation"],
                        curve["residual"][block] / curve["standard_deviation"][block],
                        color=colour,
                        ls=linestyle,
                        label=state[:4],
                    )
                    residual_axes[p, b].set(
                        title=rf"{name} / $\ell={ell}$", ylabel=r"$(d-m)/\sqrt{C_{ii}}$"
                    )
                    if p == 1:
                        ax.set_xlabel(r"$s$ [$h^{-1}{\rm Mpc}$]")
                        residual_axes[p, b].set_xlabel(r"$s$ [$h^{-1}{\rm Mpc}$]")
        axes[0, 0].legend(fontsize=7)
        residual_axes[0, 0].legend(fontsize=7)
        stem = condition.replace(".", "p")
        show(
            fig,
            "clustering-" + stem,
            condition + ": mean of all paired mock vectors. "
            "Shading is one-realization sample scatter, not error on the mean. "
            "Separations are mean pair-weighted coordinates on identical bin edges.",
        )
        show(
            residual_fig,
            "residuals-" + stem,
            condition
            + ": fixed mock "
            + str(grid["mock_ids"][0])
            + ", chosen by ID, not fit quality. Diagonal-normalized "
            "total-model residuals; neither whitened residuals nor isolated acoustic components.",
        )
    return source


if LRG_REPORTS:
    lrg_figures_curve_figures(curves, lrg_grid)


In [ ]:
# Appendix: V_eff-matched LRG2/LRG3 density tiers (sigma_z 0.001 and 0.003) against the
# per-bin primary at the same routes (revision 2.15). Both inputs are eight-command runs.
# From revision 2.20 pass the two unified-prior refits (unified-fit-v3--density and
# unified-fit-v3--primary); render_robustness.py supplies them as robustness_run_choices.
_density_choices = globals().get("robustness_run_choices") or {}
MATCHED_DENSITY_RUN = _density_choices.get("density")  # the density successor run (…--density)
PER_BIN_PRIMARY_RUN = _density_choices.get("density_primary")  # the per-bin primary it extends

density_appendix = None
if MATCHED_DENSITY_RUN is not None and PER_BIN_PRIMARY_RUN is not None:
    density_appendix = readers.matched_density_comparison(MATCHED_DENSITY_RUN, PER_BIN_PRIMARY_RUN)
    display(Markdown(
        "Rows read: %d; paired mocks: %d; single-vs-mean rows: %d" % (
            len(density_appendix["rows"]), len(density_appendix["pairs"]),
            len(density_appendix["single_vs_mean"]),
        )
    ))

In [ ]:
def density_appendix_plot(result):
    """Primary versus matched density per tier: bias, scatter, paired widths, single-vs-mean."""
    import matplotlib.pyplot as plt

    states = ("prereconstruction", "postreconstruction")
    tiers = sorted({r["condition"] for r in result["summary"]})
    colours = {"primary": "#4d4d4d", "matched": "#e41a1c"}
    routes = [(t, n, s) for t in tiers for n in ("LRG2", "LRG3") for s in states]
    labels = [f"{n}\n{t.split('_')[1]}\n{s[:4]}" for t, n, s in routes]
    for parameter in PARAMETERS:
        fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
        for offset, variant in ((-0.12, "primary"), (0.12, "matched")):
            rows = [
                next(
                    r for r in result["summary"]
                    if r["condition"] == t and r["measurement_bin"] == n
                    and r["reconstruction_state"] == s and r["parameter"] == parameter
                    and r["variant"] == variant
                )
                for t, n, s in routes
            ]
            x = np.arange(len(rows)) + offset
            bias = [np.nan if r["bias"] is None else r["bias"] for r in rows]
            sem = [np.nan if r["bias_sem"] is None else r["bias_sem"] for r in rows]
            axes[0].errorbar(x, bias, yerr=sem, fmt="o", color=colours[variant], label=variant)
            axes[1].plot(x, [np.nan if r["scatter"] is None else r["scatter"] for r in rows],
                         "s", color=colours[variant], label=variant)
            for xi, r in zip(x, rows):
                axes[1].annotate(f"{r['n_usable_interval']}/{r['n_saved']}", (xi, 0),
                                 xycoords=("data", "axes fraction"), ha="center", fontsize=7,
                                 color=colours[variant])
                if variant == "matched":
                    note = f"{100 * r['veff_fraction']:.0f}%" + (" (ceiling)" if r["bounded"] else "")
                    axes[0].annotate(note, (xi, 1), xycoords=("data", "axes fraction"),
                                     ha="center", va="bottom", fontsize=7, color=colours[variant])
        axes[0].axhline(0.0, color="k", lw=0.8)
        axes[0].set_ylabel(f"ensemble bias in {LABELS[parameter]} (mean - 1)")
        axes[1].set_ylabel(f"scatter of {LABELS[parameter]}")
        for ax in axes[:2]:
            ax.set_xticks(np.arange(len(labels)), labels, fontsize=7)
            ax.legend(frameon=False)
        ratios = [
            [p["width_ratio_matched_over_primary"] for p in result["pairs"]
             if p["condition"] == t and p["measurement_bin"] == n
             and p["reconstruction_state"] == s and p["parameter"] == parameter
             and p["usable_intervals"]]
            for t, n, s in routes
        ]
        axes[2].boxplot([r if r else [np.nan] for r in ratios], tick_labels=labels)
        axes[2].tick_params(axis="x", labelsize=7)
        axes[2].axhline(1.0, color="k", lw=0.8)
        axes[2].set_ylabel("paired interval width, matched / primary")
        single = [
            [p["width_ratio_single_matched_over_mean_primary"] for p in result["single_vs_mean"]
             if p["condition"] == t and p["measurement_bin"] == n
             and p["reconstruction_state"] == s and p["parameter"] == parameter
             and p["usable_intervals"]]
            for t, n, s in routes
        ]
        expected = [
            next((p["expected_sqrt_veff_ratio"] for p in result["single_vs_mean"]
                  if p["condition"] == t and p["measurement_bin"] == n
                  and p["reconstruction_state"] == s and p["parameter"] == parameter), np.nan)
            for t, n, s in routes
        ]
        axes[3].boxplot([r if r else [np.nan] for r in single], tick_labels=labels)
        axes[3].plot(np.arange(1, len(routes) + 1), expected, "_", color="C0", ms=14,
                     label=r"$\sqrt{V_{\rm eff}^{\rm mean}/V_{\rm eff}^{\rm matched}}$")
        axes[3].tick_params(axis="x", labelsize=7)
        axes[3].set_ylabel("width, one matched realization / mean-of-25 primary")
        axes[3].legend(frameon=False)
        show(fig, f"density_appendix_{parameter}",
             "Matched effective volume (SPHEREx 30,000 deg^2) by raised n(z) at fixed shape; the "
             "attained V_eff fraction is annotated (LRG2 at 0.003 stops at the native ceiling). "
             "Usable intervals / saved fits are annotated; failed or boundary-limited fits stay "
             "out of the ratios but are counted. The last panel compares one matched realization "
             "with the mean of 25 sparse realizations against the Gaussian-budget expectation.")


if density_appendix is not None:
    density_appendix_plot(density_appendix)


In [ ]:
# Eight-command run inputs (full-run campaign). Leave None to skip a panel; render_robustness.py
# supplies them as robustness_run_choices. From revision 2.20 the paper reads the unified-prior
# refits: the unified primary refit with its flat and PCS2 checks (one PCS2 run serves both
# tracers), the unified RecIso and scan refits and (cross-sample run, self run) pairs.
RUNS = os.path.join(os.environ.get("BAO_STORAGE_ROOT", "outputs"), "runs")
_choices = globals().get("robustness_run_choices") or {}
BASELINE_RUN = _choices.get("baseline")  # f"{RUNS}/core-sensitivity/unified-fit-v3--primary"
FLAT_RUN = _choices.get("flat")  # the baseline's flat check (…--primary--flat)
PCS2_LRG = _choices.get("pcs2_lrg")  # the baseline's PCS2 check (…--primary--pcs2)
PCS2_BGS = _choices.get("pcs2_bgs")
RECISO_RUN = _choices.get("reciso")  # f"{RUNS}/core-sensitivity/unified-fit-v3--reciso--R15"
SCAN_RUNS = list(_choices.get("scan", []))  # unified-fit-v3--scan--R{15,25,35}
TRANSFER_PAIRS = list(_choices.get("transfer", []))
# e.g. [(f"{RUNS}/reconstruction-transfer/unified-fit-v3--sigma_0.003--R25",
#        f"{RUNS}/reconstruction-transfer/unified-fit-v3--self--R25")]

run_robustness = scan_summary = reproducibility = None
transfer_results, transfer_intervals = [], []
if BASELINE_RUN is not None:
    means, _ = readers.robustness_rows_from_runs(
        BASELINE_RUN, flat=FLAT_RUN, pcs2_lrg=PCS2_LRG, pcs2_bgs=PCS2_BGS,
        reciso=RECISO_RUN, r25=next((r for r in SCAN_RUNS if str(r).endswith("R25")), None),
    )
    run_robustness = readers.robustness_grid(means)
if SCAN_RUNS:
    scan_summary = readers.radius_scan_rows(SCAN_RUNS)
    r15 = next((r for r in SCAN_RUNS if str(r).endswith("R15")), None)
    if r15 is not None and BASELINE_RUN is not None:
        reproducibility = readers.scan_reproducibility_rows(r15, BASELINE_RUN)
for variant_run, self_run in TRANSFER_PAIRS:
    # The paired comparison verifies identical targets and randoms before any interval is shown.
    transfer_results.append((variant_run, readers.transfer_comparison(variant_run, self_run)))
    transfer_intervals.extend(readers.transfer_mean_fits(variant_run, self_run))

In [ ]:
if run_robustness is not None:
    claim_figures_plot_robustness(run_robustness)


def radius_curve_plot(summary):
    """Recovery, scatter and interval width against smoothing radius, per tier and parameter."""
    import matplotlib.pyplot as plt

    tiers = sorted({r["condition"] for r in summary})
    for parameter in PARAMETERS:
        fig, axes = plt.subplots(3, len(tiers), figsize=(4.2 * len(tiers), 9),
                                 sharex=True, squeeze=False, layout="constrained")
        for column, tier in enumerate(tiers):
            for name in BINS:
                rows = sorted(
                    (r for r in summary if r["condition"] == tier and r["measurement_bin"] == name
                     and r["parameter"] == parameter and r["reconstruction_state"] == "postreconstruction"),
                    key=lambda r: r["smoothing_mpc_h"],
                )
                if not rows:
                    continue
                x = [r["smoothing_mpc_h"] for r in rows]
                nan = lambda v: np.nan if v is None else v
                axes[0, column].errorbar(x, [nan(r["bias"]) for r in rows],
                                         yerr=[nan(r["bias_sem"]) for r in rows], marker="o",
                                         color=BIN_COLORS[name], label=name)
                axes[1, column].plot(x, [nan(r["scatter"]) for r in rows], "s-", color=BIN_COLORS[name])
                axes[2, column].plot(x, [nan(r["median_width_ratio_vs_reference"]) for r in rows],
                                     "^-", color=BIN_COLORS[name])
            axes[0, column].axhline(0, color="k", lw=0.8)
            axes[2, column].axhline(1, color="k", lw=0.8)
            axes[0, column].set_title(tier)
            axes[2, column].set_xlabel(r"smoothing radius [$h^{-1}$Mpc]")
        axes[0, 0].set_ylabel(f"bias in {LABELS[parameter]}")
        axes[1, 0].set_ylabel(f"scatter of {LABELS[parameter]}")
        axes[2, 0].set_ylabel("median width / R15")
        axes[0, 0].legend(frameon=False, fontsize=8)
        show(fig, f"radius_scan_{parameter}",
             "Post-reconstruction recovery against smoothing radius under the per-bin damping "
             "priors. Failed, boundary-limited and unpaired fits stay out of the ratios but "
             "remain in the saved-fit counts.")


if scan_summary is not None:
    radius_curve_plot(scan_summary)
if reproducibility is not None:
    paired = [r for r in reproducibility if not r["missing"]]
    display(Markdown(
        "Scan R15 vs core R15: %d paired fits; max |difference| = %s; %d unpaired." % (
            len(paired),
            None if not paired else max(abs(r["scan_minus_core"]) for r in paired),
            len(reproducibility) - len(paired),
        )
    ))


def transfer_plot(results):
    """Mean paired shift (transfer minus self) with its standard error, per class and bin."""
    import matplotlib.pyplot as plt

    for variant_run, (_, summaries) in results:
        for parameter in PARAMETERS:
            rows = [s for s in summaries if s["parameter"] == parameter]
            if not rows:
                continue
            fig, ax = plt.subplots(figsize=(8, 3.6), layout="constrained")
            labels = [f"{s['measurement_bin']}\n{s['condition']}" for s in rows]
            y = [np.nan if s["mean_variant_minus_self"] is None else s["mean_variant_minus_self"]
                 for s in rows]
            e = [np.nan if s["paired_mean_standard_error"] is None else s["paired_mean_standard_error"]
                 for s in rows]
            ax.errorbar(np.arange(len(rows)), y, yerr=e, fmt="o")
            ax.axhline(0, color="k", lw=0.8)
            ax.set_xticks(np.arange(len(rows)), labels, fontsize=7)
            ax.set_ylabel(f"transfer - self, {LABELS[parameter]}")
            ax.set_title(Path(variant_run).name, fontsize=9)
            show(fig, f"transfer_{Path(variant_run).name}_{parameter}",
                 "Paired on identical targets and randoms; missing pairs keep the mean undefined. "
                 "Agreement does not establish unbiased recovery.")


if transfer_results:
    transfer_plot(transfer_results)

In [ ]:
# Paper figures for the robustness appendix and the cross-sigma_z sample result, drawn from the run
# readers above. Each producer keeps flagged fits visible (open markers) and leaves routes
# without fits blank; render_robustness.py saves the five reviewed names (two for the choices).
PAPER_VARIANTS = (
    ("pre", "Pre-reconstruction"),
    ("flat", "Flat damping priors"),
    ("pcs2", "PCS2 broadband"),
    ("reciso", "RecIso"),
    ("r25", r"$\Sigma_{\rm sm}=25\,h^{-1}{\rm Mpc}$"),
)
#: Reference bands take the light member of each redshift bin's Paired-style pair at reduced
#: opacity, so that a band reads as a pale version of its bin's markers.
ZBIN_BAND_COLORS = tuple(light for light, _ in style.ZBIN_COLOR_PAIRS)
BAND_ALPHA = 0.45


def _paper_axes_style(ax):
    ax.tick_params(axis="both", labelsize=7)
    ax.xaxis.label.set_fontsize(8)
    ax.yaxis.label.set_fontsize(8)


def _legend_handles(entries):
    from matplotlib.lines import Line2D

    return [Line2D([], [], marker="o", ms=3, lw=0.8, color=color) for color in entries]


def _interval_marker(ax, x, y, xerr, color, usable, **kwargs):
    ax.errorbar(x, y, xerr=xerr, fmt="o", ms=3, color=color, mfc=color if usable else "white",
                elinewidth=0.8, capsize=0, **kwargs)


PRIOR_RANGE = (0.8, 1.2)  # the uniform dilation prior; prior-bounded bars are clipped to it


def _forest_panel(ax, points, n_rows, bands=(), separators=()):
    """One forest panel in the style of Paillas et al. 2024 (Figs 15-16): rows top-down.

    ``points`` holds (row, value, low, high, usable, color); ``bands`` holds
    (first_row, last_row, low, high, color) reference intervals drawn as pale spans behind
    their rows.
    """
    for first, last, low, high, color in bands:
        ax.fill_betweenx([first - 0.45, last + 0.45], low, high, color=color, alpha=BAND_ALPHA,
                         lw=0, zorder=0)
    for y in separators:
        ax.axhline(y, color="0.75", lw=0.5)
    for y, value, low, high, usable, color in points:
        ax.errorbar(value, y, xerr=[[value - low], [high - value]], fmt="o", ms=3, color=color,
                    mfc=color if usable else "white", elinewidth=0.8, capsize=0, zorder=2)
    ax.axvline(1, color="0.4", lw=0.6, ls=":")
    if points:
        low = min(p[2] for p in points)
        high = max(p[3] for p in points)
        pad = 0.08 * (high - low)
        ax.set_xlim(max(low - pad, PRIOR_RANGE[0]), min(high + pad, PRIOR_RANGE[1]))
    ax.set_ylim(n_rows - 0.5, -0.5)
    ax.yaxis.set_minor_locator(plt.NullLocator())  # categories: major ticks only
    _paper_axes_style(ax)


def _points(entries, color, start=0):
    return [(start + y, *entry, color) for y, entry in enumerate(entries) if entry is not None]


def _minos_entry(entry):
    if entry is None or entry.get("status") != "available" or not entry.get("ordered_bounds"):
        return None
    low, high = entry["minos_bounds"]
    return entry["value"], low, high, entry["usable"]


def paper_robustness_choices(grid):
    """One forest figure per dilation: a panel per bin, the reference on top, then each variant."""
    lookup = {(r["measurement_bin"], r["parameter"], r["variant"]): r for r in grid}
    rows = (("baseline", "Post-recon. (reference)"),) + PAPER_VARIANTS
    for parameter, suffix in zip(PARAMETERS, ("iso", "ap")):
        reference = {name: _minos_entry(lookup.get((name, parameter, "baseline"))) for name in BINS}
        # A bin whose reference interval is prior-bounded carries no alpha_AP information.
        shown = [b for b, name in enumerate(BINS)
                 if parameter == "qiso" or (reference[name] is not None and reference[name][3])]
        if not shown:
            continue
        with plt.style.context(style.PRD_STYLE_FILE):
            fig, axes = plt.subplots(1, len(shown), figsize=style.two_col(2.3), sharey=True,
                                     squeeze=False, layout="constrained")
            for ax, b in zip(axes[0], shown):
                name = BINS[b]
                entries = [_minos_entry(lookup.get((name, parameter, v))) for v, _ in rows]
                bands = [] if reference[name] is None else [
                    (0, len(rows) - 1, *reference[name][1:3], ZBIN_BAND_COLORS[b])
                ]
                _forest_panel(ax, _points(entries, style.zbin_color(b)), len(rows), bands)
                ax.set_title(ZLABELS[b], fontsize=7)
                ax.set_xlabel(LABELS[parameter])
            axes[0, 0].set_yticks(range(len(rows)), [label for _, label in rows], fontsize=6.5)
        show(fig, f"fig_robustness_choices_{suffix}",
             f"{LABELS[parameter]}: mean of 25 realizations with C/25 at sigma_0.003 after "
             "reconstruction, reference (top, shaded band) and each variant per redshift bin; open "
             "markers flag unusable intervals.")


def paper_robustness_smoothing(summary):
    """Forest of the post-reconstruction mean fit against the smoothing radius.

    Columns are the photo-z conditions and the two panel rows the dilations; within a panel each
    redshift bin is a group of radii, with its 15 Mpc/h interval as the shaded reference band.
    """
    tiers = [t for t in TIERS if any(r["condition"] == t for r in summary)]
    radii = sorted({r["smoothing_mpc_h"] for r in summary})
    lookup = {
        (r["smoothing_mpc_h"], r["condition"], r["measurement_bin"], r["parameter"]): r
        for r in summary if r["reconstruction_state"] == "postreconstruction"
    }

    def entry(r):
        if r is None or r["mean_fit_value"] is None:
            return None
        value, half = r["mean_fit_value"], r["mean_fit_half_width"]
        return value, value - half, value + half, r["mean_fit_usable"]

    missing = sorted({(radius, name) for radius in radii for name in BINS for t in tiers
                      if (lookup.get((radius, t, name, "qiso")) or {}).get("mean_fit_value") is None})
    n = len(radii)
    with plt.style.context(style.PRD_STYLE_FILE):
        fig, axes = plt.subplots(len(PARAMETERS), len(tiers), figsize=style.two_col(5.2),
                                 sharey=True, squeeze=False, layout="constrained")
        for column, tier in enumerate(tiers):
            for row, parameter in enumerate(PARAMETERS):
                ax = axes[row, column]
                points, bands = [], []
                for b, name in enumerate(BINS):
                    rows = [entry(lookup.get((radius, tier, name, parameter))) for radius in radii]
                    points += _points(rows, style.zbin_color(b), start=b * n)
                    if rows[0] is not None:
                        bands.append((b * n, b * n + n - 1, *rows[0][1:3], ZBIN_BAND_COLORS[b]))
                _forest_panel(ax, points, len(BINS) * n, bands,
                              separators=[b * n - 0.5 for b in range(1, len(BINS))])
                if column == 0:
                    ax.set_yticks(range(len(BINS) * n),
                                  [f"{radius:g}" for _ in BINS for radius in radii], fontsize=6)
                ax.set_xlabel(LABELS[parameter])
            axes[0, column].set_title(rf"$\sigma_0={tier.split('_')[1]}$", fontsize=8)
        fig.supylabel(r"$\Sigma_{\rm sm}$ [$h^{-1}{\rm Mpc}$]", fontsize=7)
        fig.legend(_legend_handles(style.zbin_color(b) for b in range(len(BINS))), ZLABELS,
                   fontsize=6.5, frameon=False, loc="outside upper center", ncol=len(BINS))
    note = "" if not missing else " Without mean fits yet: " + ", ".join(
        f"R{radius:g} {name}" for radius, name in missing) + "."
    show(fig, "fig_robustness_smoothing",
         "Post-reconstruction mean fit of 25 realizations (C/25) with its Minos interval against "
         "the smoothing radius, per redshift bin (groups) and photo-z condition (columns); the shaded "
         "band is each bin's 15 Mpc/h interval. Open markers: flagged mean-fit intervals." + note)


def paper_robustness_transfer(rows):
    """Forest of alpha_iso for the target classes: a panel per bin and per target condition."""
    states = (("self", "Self-reconstruction (reference)"),
              ("cross", r"Cross-$\sigma_z$ sample, $\sigma_0=0.003$ source"),
              ("pre", "Pre-reconstruction"))
    conditions = sorted({r["condition"] for r in rows}, key=TIERS.index)
    lookup = {(r["source_kind"], r["measurement_bin"], r["condition"], r["parameter"]): r
              for r in rows}

    def entry(r):
        if r is None or r["status"] != "complete" or r["half_width"] is None:
            return None
        return r["value"], r["value"] - r["half_width"], r["value"] + r["half_width"], r["usable"]

    with plt.style.context(style.PRD_STYLE_FILE):
        fig, axes = plt.subplots(len(conditions), len(BINS), figsize=style.two_col(3.0),
                                 sharey=True, squeeze=False, layout="constrained")
        for row, condition in enumerate(conditions):
            for b, name in enumerate(BINS):
                ax = axes[row, b]
                entries = [entry(lookup.get((kind, name, condition, "qiso"))) for kind, _ in states]
                bands = [] if entries[0] is None else [
                    (0, len(states) - 1, *entries[0][1:3], ZBIN_BAND_COLORS[b])
                ]
                _forest_panel(ax, _points(entries, style.zbin_color(b)), len(states), bands)
                if row == 0:
                    ax.set_title(ZLABELS[b], fontsize=7)
                if row == len(conditions) - 1:
                    ax.set_xlabel(LABELS["qiso"])
            axes[row, 0].set_yticks(range(len(states)), [label for _, label in states], fontsize=6.5)
            axes[row, -1].text(1.04, 0.5, rf"target $\sigma_0={condition.split('_')[1]}$",
                               transform=axes[row, -1].transAxes, rotation=270, ha="left",
                               va="center", fontsize=7)
    show(fig, "fig_robustness_transfer",
         "alpha_iso from the mean of 25 realizations with C/25 for the sigma_0.01 and 0.03 target "
         "classes at 25 Mpc/h: self-reconstruction (shaded band, all rows), cross-sigma_z sample and pre-reconstruction; "
         "open markers: flagged intervals. alpha_AP reaches the prior bounds and is not shown.")


def paper_robustness_density(result):
    """Mean-fit intervals at the target and the matched density, before and after reconstruction."""
    keys = sorted({(r["measurement_bin"], r["condition"]) for r in result["summary"]})
    lookup = {(r["variant"], r["measurement_bin"], r["condition"], r["reconstruction_state"],
               r["parameter"]): r for r in result["summary"]}
    target, matched = style.categorical_pair(0), style.categorical_pair(1)
    series = (
        ("primary", "prereconstruction", target[0], "Target density, pre"),
        ("primary", "postreconstruction", target[1], "Target density, post"),
        ("matched", "prereconstruction", matched[0], r"Matched $V_{\rm eff}$, pre"),
        ("matched", "postreconstruction", matched[1], r"Matched $V_{\rm eff}$, post"),
    )
    with plt.style.context(style.PRD_STYLE_FILE):
        fig, axes = plt.subplots(1, 2, figsize=style.one_col(3.0), sharey=True,
                                 layout="constrained")
        for ax, parameter in zip(axes, PARAMETERS):
            for y, key in enumerate(keys):
                for offset, (variant, state, color, _) in zip((-0.27, -0.09, 0.09, 0.27), series):
                    r = lookup.get((variant, *key, state, parameter))
                    if r is None or r["mean_fit_value"] is None:
                        continue
                    _interval_marker(ax, r["mean_fit_value"], y + offset, r["mean_fit_half_width"],
                                     color, r["mean_fit_usable"])
            ax.axvline(1, color="0.4", lw=0.6)
            ax.set_xlabel(LABELS[parameter])
            _paper_axes_style(ax)
        # Rows are named by redshift range and redshift precision.
        axes[0].set_yticks(
            range(len(keys)),
            [f"{ZLABELS[BINS.index(name)]}\n$\\sigma_z/(1+z)={condition.split('_')[1]}$"
             for name, condition in keys],
            fontsize=6.5,
        )
        axes[0].set_ylim(len(keys) - 0.5, -0.5)
        axes[0].yaxis.set_minor_locator(plt.NullLocator())  # categories: major ticks only
        fig.legend(_legend_handles(color for _, _, color, _ in series),
                   [label for *_, label in series], fontsize=6, frameon=False,
                   loc="outside upper center", ncol=2)
    show(fig, "fig_robustness_density",
         "Mean of 25 realizations with C/25 before (light) and after (dark) reconstruction; open "
         "markers: flagged intervals. The matched samples keep the radial shape of the mocks.")


if run_robustness is not None:
    paper_robustness_choices(run_robustness)
if scan_summary is not None:
    paper_robustness_smoothing(scan_summary)
if transfer_intervals:
    paper_robustness_transfer(transfer_intervals)
if density_appendix is not None:
    paper_robustness_density(density_appendix)

## Save figures

Choose an output directory under `products/` to save PNG and PDF copies. Rerunning this cell updates those figures.


In [ ]:
# Set a directory when you want to save the plots currently in figures.
OUTPUT_DIR = None  # REPO / "products" / "figure-exploration"
if OUTPUT_DIR is not None:
    destination = Path(OUTPUT_DIR)
    destination.mkdir(parents=True, exist_ok=True)
    for name, fig in figures.items():
        fig.savefig(destination / f"{name}.png", dpi=300, bbox_inches="tight")
        fig.savefig(destination / f"{name}.pdf", bbox_inches="tight")
